In [21]:
import numpy as np
import pandas as pd
import h5py

from astropy.table import Table
import glob

In [7]:
snapshot_list = [i for i in range(87, 99)]  # Your list of snapshot numbers

dfs = []
with h5py.File('stellar_circs.hdf5', 'r') as f:
    for snapshot_n in snapshot_list:
        data = np.array(f[f'Snapshot_{snapshot_n}']['MassTensorEigenVals'])
        names = np.array(f[f'Snapshot_{snapshot_n}']['SubfindID'])
        
        df = pd.DataFrame({
            'snapshot_n': snapshot_n,
            'SubfindID': names,
            'c_axis': data[:, 0],
            'b_axis': data[:, 1],
            'a_axis': data[:, 2]
        })
        dfs.append(df)

result_df = pd.concat(dfs, ignore_index=True)
result_df

,snapshot_n,SubfindID,c_axis,b_axis,a_axis
0,87,0,29.131363,35.139507,56.422882
1,87,1,2.046793,2.745060,2.895154
2,87,2,2.673068,4.737928,4.770403
3,87,3,0.776014,1.204350,1.425162
4,87,4,0.667322,0.888832,1.291917
...,...,...,...,...,...
93804,98,1008805,1.114021,1.173031,1.192116
93805,98,1009697,0.858958,0.887657,0.931893
93806,98,1013786,0.750311,0.781825,0.807100
93807,98,1035619,1.020942,1.037153,1.081229


In [15]:
result_df['Name'] = result_df.apply(lambda x: f'{x.snapshot_n:.0f}_{x.SubfindID:.0f}', axis=1)

In [16]:
mangia_table = Table.read('MaNGIA_catalog.fits', format='fits').to_pandas()

In [31]:
mangia_table['Name'] = mangia_table.apply(lambda x: f'{x.snapshot:.0f}_{x.subhalo_id:.0f}', axis=1)
mangia_table['Name_ext'] = mangia_table.apply(lambda x: f'{x.snapshot:.0f}_{x.subhalo_id:.0f}_{x['view']}', axis=1)

In [32]:
joined = pd.merge(result_df, mangia_table, on='Name', how='inner', suffixes=('', '_mangia'))

In [33]:
joined

,snapshot_n,SubfindID,c_axis,b_axis,a_axis,Name,snapshot,subhalo_id,view,stellar_mass,re_kpc,sample_manga,manga_ifu_dsn,distances_selec,n_star_part,n_gas_cell,Name_ext
0,87,141934,1.586863,4.391834,4.618956,87_141934,87,141934,0,11.153300,7.519479,2,37,0.178884,2434971,55565,87_141934_0
1,87,155298,3.720521,5.670828,5.912961,87_155298,87,155298,0,11.286005,12.157663,1,61,0.343986,3710124,551798,87_155298_0
2,87,192324,3.903698,5.537130,5.598183,87_192324,87,192324,0,11.317500,12.692906,1,61,0.258038,3923066,1443785,87_192324_0
3,87,230905,3.258412,6.851683,7.190629,87_230905,87,230905,0,11.041639,14.455937,2,127,0.223563,2001520,378145,87_230905_0
4,87,266847,1.846751,4.302204,4.315521,87_266847,87,266847,0,11.132612,8.187306,2,37,0.039083,2432255,224681,87_266847_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10046,98,804001,0.463718,0.626417,0.635941,98_804001,98,804001,0,8.583290,1.507643,-1,61,0.027776,6700,17539,98_804001_0
10047,98,813098,0.254221,0.317876,0.320544,98_813098,98,813098,0,8.821639,0.989591,1,37,0.039944,13832,8353,98_813098_0
10048,98,818866,0.482917,0.616868,0.620115,98_818866,98,818866,0,8.852764,1.836591,1,91,0.054014,14479,0,98_818866_0
10049,98,851110,0.200966,0.226862,0.246531,98_851110,98,851110,0,8.818266,0.841721,1,19,0.038219,16680,0,98_851110_0


In [25]:
dir_name = 'VELOCITY_VDISP_FLUX MAPS'
files_list = glob.glob(f'{dir_name}/TNG50*.fits')

In [26]:
files_list = [x.split('/')[-1] for x in files_list]

In [ ]:
ids = [x.split('-')[2] for x in files_list]
n_snap = [x.split('-')[1] for x in files_list]
n_view = [x.split('-')[3] for x in files_list]

In [37]:
names = [x + '_' + y for x, y in zip(n_snap, ids)]
names_ex = [x + '_' + y + '_' + z for x, y, z in zip(n_snap, ids, n_view)]

In [38]:
ifu_files = pd.DataFrame({'filename': files_list, 'sub_id': ids, 'snapshot': n_snap, 'view': n_view, 'Name': names, 'Name_ext': names_ex})

In [39]:
joined = pd.merge(ifu_files, joined, on='Name_ext', how='left', suffixes=('', '_joined'))

In [41]:
joined.to_csv('files_list_and_axis.csv', index=False)